# 07 - Discovered indicators

The indicators the network learned, drawn on the price of one window next to the same indicators at their
textbook periods: the discovered indicators are what a run delivers (VISION), and the prediction and trading
quality in 01-04 are the evidence that they are good. Each copy of every registered family is drawn twice on the
same window: solid at the period the served model applied to that window, dashed at its configured textbook
period. The families are moving average, MACD, RSI, Bollinger, ATR, stochastic, Williams %R, Keltner, OBV,
VWAP, MFI, ADX, CCI and Donchian. The original four use the numpy formulas on the raw close; the other ten are
drawn from each family's outputs on the model window. Lines that smooth use an EWMA with alpha = 2 / (period + 1),
started at the window's first bar.

Set `WINDOW` to a window index, or to "typical" (closest to the block's median periods), "longest" / "shortest"
(where the per-window adjustment stretches or shrinks the periods most) or "last"; `BLOCK` to another block of
the run's fold. Nothing here writes to the run.

In [1]:
# Parameters
RUN_DIR = None                  # a runs/<id> directory; None -> the newest run under RUNS_DIR
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
BLOCK = "test"                  # "train", "val", "cal" or "test"
WINDOW = "typical"              # a window index, or "typical" / "longest" / "shortest" / "last"

In [2]:
from neural_trade.data.processor import split_arrays
from neural_trade.notebook import pick_run
from neural_trade.registries.visualizations import Visualizations
from neural_trade.serving.predictor import Predictor
from neural_trade.visualization import analytics_tables as AT
from neural_trade.visualization.discovered_indicators import discovered_table
from neural_trade.visualization.indicator_evolution import applied_periods

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
predictor = Predictor.from_artifacts(run_dir / "artifacts")
cfg = predictor.config.copy(CSV_PATH=CSV_PATH)
blocks = split_arrays(cfg)
block = blocks[BLOCK]
times = blocks["df"]["timestamp"].to_numpy()[block["anchor_bar"]]
metrics = run_dir / "metrics.jsonl"
applied = applied_periods(predictor, block["X_model"], block=BLOCK)   # the period each window gets (served weights)
print(f"{len(block['X']):,} {BLOCK} windows of {cfg.LOOKBACK} bars; served weights: epoch",
      predictor.bundle.meta.get("weights_epoch"))

run: ..\runs\20261004T073704Z-d1f6fa9-11993eec
CalibrationPipeline loaded from '..\runs\20261004T073704Z-d1f6fa9-11993eec\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


7,236 test windows of 60 bars; served weights: epoch 10


## The learned indicators on price

Top: the block's close, one point per window, with the drawn window shaded. The grid: one row per family, one
column per copy; each panel's heading gives the learned and the textbook period (MACD: fast / slow / signal).
Below it, each base period over training (from the recorded start, the dashed line is its textbook value) and,
right of the last epoch, how the period the model applies varies across the block's windows: 5-95% and the middle
50% of the windows, their median, this window's period and the served base period. A period longer than the
window has not warmed up by the window's end (the EWMA starts at the window's first bar); the table's
"applied > lookback" column counts those windows.

In [3]:
Visualizations.build("discovered_indicators", block["X"], cfg, applied=applied, metrics=metrics, window=WINDOW,
                     times=times, ohlcv=block["X_model"]).show()

## Learned against textbook periods

The table behind the figure, in bars: the served base period (the trained logit alone) and the median applied
period, each with its change against the textbook period; the applied range over the block's windows; this
window's period; the range the base period took during training; the share of windows whose applied period is
longer than the window; and whether the base period sits at a clip bound.

In [4]:
AT.styled(discovered_table(applied, cfg, metrics=metrics, window=WINDOW), digits=2)

,indicator,textbook,learned (served base),base vs textbook %,applied median,median vs textbook %,applied p5,applied p95,this window,"applied > lookback, % of windows",training min,training max,base near a clip bound
period,,,,,,,,,,,,,
ma_period_0,MA #0,5.00,4.94,-1.16,5.65,12.91,4.63,6.84,5.84,0.00,4.65,5.34,
ma_period_1,MA #1,10.00,9.13,-8.72,6.12,-38.76,5.93,7.89,6.11,0.00,8.94,11.48,
ma_period_2,MA #2,30.00,43.20,44.01,69.51,131.68,55.34,70.58,69.61,89.04,30.00,49.94,
macd_0_fast,MACD #0 fast,12.00,8.86,-26.18,5.88,-50.96,5.77,7.15,5.87,0.00,8.57,12.00,
macd_0_slow,MACD #0 slow,26.00,43.37,66.82,53.70,106.54,45.05,68.21,54.44,27.64,25.54,51.49,
macd_0_signal,MACD #0 signal,9.00,11.20,24.44,12.32,36.90,8.11,17.18,12.10,0.00,8.62,12.33,
macd_1_fast,MACD #1 fast,5.00,5.14,2.78,4.35,-13.06,3.55,5.92,4.36,0.00,3.79,5.48,
macd_1_slow,MACD #1 slow,35.00,40.11,14.61,26.16,-25.26,24.74,32.92,26.05,0.00,35.00,52.97,
macd_1_signal,MACD #1 signal,5.00,7.04,40.83,8.96,79.15,7.24,10.72,9.07,0.00,4.68,7.72,
